<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/Dataset_Manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wearable Biomechanics Dataset Manager


#### Setup Environment

In [ ]:
# @title
!pip install duckdb pyarrow pandas


In [6]:
# @title
import duckdb
import pandas as pd
from IPython.display import display
from google.colab import drive
import json
import os
from datetime import datetime

In [7]:
# @title
drive.mount('/content/drive', force_remount=True)
BASE_DIR = '/content/drive/MyDrive/Datasets/Wearable-Biomechanics'

Mounted at /content/drive


# Subject Manager


In [18]:
# @title SubjectManager
class SubjectManager:
    def __init__(self):
        # This targets the folder/shortcut in the mounted Drive
        self.base_dir = BASE_DIR
        self.file_path = f'{self.base_dir}/subjects.json'

        # Safety check: Create the folder if it somehow doesn't exist
        os.makedirs(self.base_dir, exist_ok=True)

        # Safety check: if the file doesn't exist, create an empty one
        if not os.path.exists(self.file_path):
            with open(self.file_path, 'w') as f:
                json.dump({}, f)
            print(f"Created a new subjects.json file in {self.base_dir}.")
        else:
            print(f"Connected to existing subjects.json in {self.base_dir}.")

    # --- HELPER FUNCTIONS ---
    def _load_data(self):
        with open(self.file_path, 'r') as f:
            return json.load(f)

    def _save_data(self, data):
        with open(self.file_path, 'w') as f:
            json.dump(data, f, indent=4)

    # --- 1. CREATE ---
    def add_subject(self, subject_id, name, weight, height, max_lift, description):
        data = self._load_data()

        if subject_id in data:
            print(f"⚠️ Error: Subject ID '{subject_id}' already exists!")
            return

        data[subject_id] = {
            "name": name,
            "weight_kg": weight,
            "height_cm": height,
            "max_lift_kg": max_lift,
            "description": description
        }
        self._save_data(data)
        print(f"✅ Successfully added {name} ({subject_id}).")

    # --- 2. READ ---
    def list_subjects(self, subject_id=None):
        data = self._load_data()

        if subject_id:
            return data.get(subject_id, "⚠️ Subject not found.")

        print(f"\n--- Database contains {len(data)} subjects ---")
        # for sub_id, info in data.items():
        #     print(f"ID: {sub_id} | Name: {info['name']} | Weight: {info['weight_kg']}kg")
        return data

    # --- 3. UPDATE ---
    def update_subject(self, subject_id, **kwargs):
        data = self._load_data()

        if subject_id not in data:
            print(f"⚠️ Error: Subject ID '{subject_id}' not found.")
            return

        for key, value in kwargs.items():
            data[subject_id][key] = value

        self._save_data(data)
        print(f"✅ Successfully updated {subject_id}.")

    # --- 4. DELETE ---
    def delete_subject(self, subject_id):
        data = self._load_data()

        if subject_id in data:
            del data[subject_id]
            self._save_data(data)
            print(f"🗑️ Successfully deleted {subject_id}.")
        else:
            print(f"⚠️ Error: Subject ID '{subject_id}' not found.")

# Initialize the manager
db = SubjectManager()

Connected to existing subjects.json in /content/drive/MyDrive/Datasets/Wearable-Biomechanics.


In [22]:
# @title List All Subjects
raw_data = db.list_subjects()
df = pd.DataFrame.from_dict(raw_data, orient='index')
df.reset_index(inplace=True)
df.rename(columns={'index': 'Subject ID', 'name': 'Name', 'weight_kg': 'Weight (kg)',
                   'height_cm': 'Height (cm)', 'max_lift_kg': 'Max Lift (kg)',
                   'description': 'Description'}, inplace=True)
display(df)


--- Database contains 0 subjects ---


,Subject ID


In [ ]:
# @title Helper Codes
db.add_subject(
    subject_id="sub_001",
    name="John Doe",
    weight=78.5,
    height=180,
    max_lift=120,
    description="Right-handed"
)
db.update_subject(
    "sub_001",
    name="John Doe",
    weight=78.5,
    height=180,
    max_lift=120,
    description="Right-handed"
)
db.delete_subject("sub_001")

🗑️ Successfully deleted sub_001.


# Experiment Manager